# RWB: ансамбль v3 + недельная модель

Ноутбук воспроизводит **`ensemble_v3_weekly50.csv`**: среднее двух прогнозов с долями 0,5 / 0,5, сопоставленных по `id`.

Здесь можно пересчитать локальные метрики из сохранённых проверочных прогнозов и собрать те же 8000 строк для платформы. Полное обучение компонентов описано по ссылкам в конце; этот запуск использует готовые прогнозы и не требует GPU. Публичная оценка смеси пока неизвестна.

## Запуск

Откройте ноутбук из клона ветки **`develop`** и выполните **Run All**. Все входы лежат в Git: `notebooks/inputs/ensemble_v3_weekly50/`, исходный test — в `data/raw/`. Для передачи команде подходит GitHub-ссылка на ноутбук вместе с доступом к репозиторию.

Установка из корня репозитория:

```bash
python -m pip install -r requirements.txt
python -m jupyterlab notebooks/ensemble_v3_weekly50.ipynb
```

Ниже не выполняются скачивание файлов, переобучение моделей или отправка на платформу.

In [1]:
from pathlib import Path
import hashlib
import importlib.util
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/rwb_forecasting").is_dir()
             and (p / "notebooks/inputs/ensemble_v3_weekly50/manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри клона develop вместе с notebooks/inputs.")

INPUTS = ROOT / "notebooks/inputs/ensemble_v3_weekly50"
manifest = json.loads((INPUTS / "manifest.json").read_text())

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def read_forecast(path):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="The 'float_precision' argument is deprecated.*")
        return pd.read_csv(path, float_precision="round_trip")

for name, record in manifest["files"].items():
    assert sha256(INPUTS / name) == record["sha256"], f"Изменён вход: {name}"
for record in (manifest["test"], manifest["blend_script"]):
    assert sha256(ROOT / record["path"]) == record["sha256"], f"Изменён файл: {record['path']}"

sys.path.insert(0, str(ROOT / "src"))
from rwb_forecasting.metrics import evaluate

spec = importlib.util.spec_from_file_location("fixed_peer_weekly_blend", ROOT / manifest["blend_script"]["path"])
blend_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(blend_module)
print("Контрольные суммы входов и скрипта сборки совпали.")

Контрольные суммы входов и скрипта сборки совпали.


## Из чего состоит решение

**V3 сокомандника:** прямые модели на восемь горизонтов. Профили дня за 28 дней сглаживаются по всем 1000 маршрутам SVD ранга 12 и входят в признаки и сезонную базу. Ансамбль: XGBoost с тремя seed / Huber / Tweedie = 0,8 / 0,1 / 0,1, затем общий калибровочный множитель.

**Недельная модель:** LightGBM прогнозирует непересекающиеся получасовые объёмы, из которых складывается часовой таргет. Модель уточняет доли маршрутов в прогнозе с признаками движения между этапами склада, сохраняя суммарный объём каждого горизонта.

**Итог:** `0.5 * v3 + 0.5 * weekly` для одного и того же `id`. Дополнительная калибровка смеси не применяется.

## Пересчёт локальных метрик

Метрика — WAPE + абсолютное относительное смещение; меньше — лучше. На каждой дате считаем её по всем 8000 значениям, затем усредняем даты с одинаковым весом.

- **Разработка, 18–31 октября 2025:** 14 дат, использованных при выборе решения. Множитель v3 = 1,0075 подобран на этих датах. Число не является независимой оценкой калибратора.
- **Исторический контроль, 27 сентября–3 октября 2025:** семь дат после фиксации смеси. Компоненты переобучались на соответствующих префиксах; множитель v3 = 1,0175 взят только из 13–26 сентября. Рецепт выбирался на октябрьской разработке, поэтому это проверка устойчивости на более раннем периоде.

NPZ ниже содержит готовые прогнозы и известный таргет из train. Код пересчитывает метрики, но не обучает эти модели заново. Контрольный XGBoost обучен на CPU; исходный конкурсный прогноз v3 — на CUDA.

In [2]:
bundles = {}
for period, count, start in [("development", 14, "2025-10-18 10:30"),
                             ("control", 7, "2025-09-27 10:30")]:
    with np.load(INPUTS / f"{period}.npz", allow_pickle=False) as archive:
        bundle = {name: archive[name].copy() for name in archive.files}
    expected_origins = np.array([str(t) for t in pd.date_range(start, periods=count, freq="D")])
    np.testing.assert_array_equal(bundle["origins"], expected_origins)
    np.testing.assert_array_equal(bundle["routes"], np.arange(1000))
    np.testing.assert_array_equal(bundle["horizon_steps"], np.arange(1, 9))
    for name in ("truth", "peer", "weekly"):
        values = bundle[name]
        assert values.shape == (count, 1000, 8)
        assert np.isfinite(values).all() and (values >= 0).all()
    bundles[period] = bundle

rows = []
for period, bundle in bundles.items():
    forecasts = {"teammate_v3": bundle["peer"],
                 "weekly_v3_latent_half_shares": bundle["weekly"],
                 "ensemble_v3_weekly50": 0.5 * bundle["peer"] + 0.5 * bundle["weekly"]}
    for model, prediction in forecasts.items():
        for i, origin in enumerate(bundle["origins"]):
            rows.append({"period": period, "origin": origin, "model": model,
                         **evaluate(bundle["truth"][i], prediction[i])})
folds = pd.DataFrame(rows)
summary = folds.groupby(["model", "period"])[["score", "wape", "abs_bias"]].mean()
display(summary["score"].unstack("period").reindex(columns=["development", "control"]).rename(columns={
    "development": "Разработка: 14 дат", "control": "Контроль: 7 дат"
}).style.format("{:.6f}"))

period,Разработка: 14 дат,Контроль: 7 дат
model,,
ensemble_v3_weekly50,0.324396,0.337244
teammate_v3,0.324858,0.339181
weekly_v3_latent_half_shares,0.326069,0.337069


In [3]:
# Сверка с опубликованными результатами предыдущих прогонов.
results = ROOT / "results/ensemble_v3_weekly50"
expected_development = json.loads((results / "metadata.json").read_text())["development_means"]
expected_development = {"ensemble_v3_weekly50" if name == "v3_weekly50" else name: value
                        for name, value in expected_development.items()}
expected_control = json.loads((results / "control/metadata.json").read_text())["means"]
for period, expected in [("development", expected_development), ("control", expected_control)]:
    for model, score in expected.items():
        assert abs(float(summary.loc[(model, period), "score"]) - score) < 1e-12

control = folds[folds.period.eq("control")].pivot(index="origin", columns="model", values="score")
wins = int((control.ensemble_v3_weekly50 < control.teammate_v3).sum())
print(f"Метрики совпали с результатами в Git. Смесь выиграла у v3 на {wins} из 7 контрольных дат.")
display(control.style.format("{:.6f}"))

Метрики совпали с результатами в Git. Смесь выиграла у v3 на 4 из 7 контрольных дат.


model,ensemble_v3_weekly50,teammate_v3,weekly_v3_latent_half_shares
origin,,,
2025-09-27 10:30:00,0.319286,0.318722,0.321006
2025-09-28 10:30:00,0.362019,0.365650,0.359600
2025-09-29 10:30:00,0.354105,0.347175,0.362629
2025-09-30 10:30:00,0.332694,0.332099,0.334624
2025-10-01 10:30:00,0.358172,0.364436,0.353359
2025-10-02 10:30:00,0.312233,0.317392,0.311232
2025-10-03 10:30:00,0.322201,0.328790,0.317036


Смесь улучшает среднее v3 на обоих периодах. На контрольной неделе отдельная недельная модель немного сильнее смеси. На этих локальных датах score смеси выше 0,32. Публичная оценка смеси пока неизвестна.

## Сборка конкурсного CSV

Используем исходный test для сопоставления ID. Проверяются полнота покрытия, отсутствие повторов, конечные неотрицательные прогнозы и контрольная сумма результата. Исходные CSV компонентов включены рядом с ноутбуком без изменения байтов.

In [4]:
test = pd.read_parquet(ROOT / manifest["test"]["path"], columns=["id", "route_id", "timestamp"])
peer = read_forecast(INPUTS / "teammate_v3.csv")
weekly = read_forecast(INPUTS / "weekly.csv")
candidate = blend_module.blend(test, peer, weekly)
assert len(candidate) == manifest["output"]["rows"] == 8000
assert candidate.columns.tolist() == ["id", "y_pred"]
display(candidate.head())

,id,y_pred
0,0,109979.572590
1,1,135192.784764
2,2,163650.613685
3,3,159728.214893
4,4,151541.751138


In [5]:
output = ROOT / manifest["output"]["path"]
if output.exists():
    assert sha256(output) == manifest["output"]["sha256"], (
        "CSV с таким именем уже существует и отличается. Сохраните его отдельно перед сборкой."
    )
else:
    command = [sys.executable, str(ROOT / manifest["blend_script"]["path"]),
               "--peer-csv", str(INPUTS / "teammate_v3.csv"),
               "--weekly-csv", str(INPUTS / "weekly.csv"),
               "--test", str(ROOT / manifest["test"]["path"]), "--output", str(output)]
    completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if completed.returncode:
        raise RuntimeError(completed.stderr or completed.stdout)

assert sha256(output) == manifest["output"]["sha256"]
restored = read_forecast(output)
np.testing.assert_array_equal(restored.id.to_numpy(), candidate.id.to_numpy())
np.testing.assert_array_equal(restored.y_pred.to_numpy(), candidate.y_pred.to_numpy())
print(f"Готово: {output.relative_to(ROOT)} — {len(restored)} строк")
print(f"SHA-256: {sha256(output)}")
print("Файл совпадает с проверенной посылкой побайтно. Загрузка на платформу выполняется вручную.")

Готово: submissions/ensemble_v3_weekly50.csv — 8000 строк
SHA-256: e24f49a8cf0c748db20c92392f64c8735ed7857a00b9501c0fefd41bcc3ee484
Файл совпадает с проверенной посылкой побайтно. Загрузка на платформу выполняется вручную.


## Полное обучение компонентов

- [V3 сокомандника: исходный notebook и режим RETRAIN](https://github.com/L3iHS/rwb-shipment-forecasting/blob/3b7394a0afeab51ef7a819164c742bf5405c387d/notebooks/solution_v3.ipynb).
- [Недельный бустинг: устройство и команда обучения](../docs/weekly_latent.md).
- [Базовый прогноз движения между этапами](../docs/stage-flow.md).
- [Методика и результаты смеси](../docs/peer-v3-blend.md).

Сохранённые проверочные прогнозы — завершённые результаты конкретных прогонов, не обученные веса моделей. Для нового варианта нужно отдельно обучать изменённый компонент и проверять его по времени. Общие обсуждения и исследовательские записи в этот notebook не включены.